In [43]:
import numpy as np

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences



# Dataset

english_sentences = [
   "hello",
    "good morning",
    "good evening",
    "i am hungry",
    "i need water",
    "i love you",
    "the cat is sleeping",
    "learning ai is fun"
]

german_sentences = [
   "hallo",
    "guten morgen",
    "guten abend",
    "ich habe hunger",
    "ich brauche wasser",
    "ich liebe dich",
    "die katze schlaeft",
    "ki lernen macht spass"
]


In [44]:


# Tokenization

eng_tokenizer = Tokenizer()
ger_tokenizer = Tokenizer()

eng_tokenizer.fit_on_texts(english_sentences)
ger_tokenizer.fit_on_texts(german_sentences)

X = eng_tokenizer.texts_to_sequences(english_sentences)
y = ger_tokenizer.texts_to_sequences(german_sentences)


In [45]:

# Padding

max_len = max(
    max(len(seq) for seq in X),
    max(len(seq) for seq in y)
)

X = pad_sequences(X, maxlen=max_len, padding="post")
y = pad_sequences(y, maxlen=max_len, padding="post")



# Only first German word : VERY IMPORTANT for this simple model


y_single = y[:, 0]
y_single = y_single.reshape(-1)



In [46]:

# Vocabulary sizes
eng_vocab_size = len(eng_tokenizer.word_index) + 1
ger_vocab_size = len(ger_tokenizer.word_index) + 1



In [47]:


# Build model
model = Sequential()

model.add(
    Embedding(
        input_dim=eng_vocab_size,
        output_dim=8,
        input_length=max_len
    )
)

model.add(
    SimpleRNN(16)
)

model.add(
    Dense(
        ger_vocab_size,
        activation="softmax"
    )
)


# Compile

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)



# Train
model.fit(
    X,
    y_single,
    epochs=300,
    batch_size=2
)




Epoch 1/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0000e+00 - loss: 2.9834  
Epoch 2/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0000e+00 - loss: 2.9413
Epoch 3/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2500 - loss: 2.9013    
Epoch 4/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3750 - loss: 2.8646
Epoch 5/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3750 - loss: 2.8250
Epoch 6/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3750 - loss: 2.7815
Epoch 7/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5000 - loss: 2.7326    
Epoch 8/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5000 - loss: 2.6864
Epoch 9/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6250 - loss: 2.6238    
Epoch 10/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6250 - loss: 2.5641
Epoch 11/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6250 - loss: 2.4991
Epoch 12/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0

In [48]:

# Reverse dictionary
reverse_german_index = {
    index: word
    for word, index in ger_tokenizer.word_index.items()
}


# Prediction function
def translate_first_word(sentence):

    sequence = eng_tokenizer.texts_to_sequences([sentence])

    sequence = pad_sequences(
        sequence,
        maxlen=max_len,
        padding="post"
    )

    prediction = model.predict(sequence)

    predicted_id = np.argmax(prediction, axis=1)[0]

    return reverse_german_index.get(predicted_id)


In [49]:

# Test

print(translate_first_word("hello"))
print(translate_first_word("good morning"))
print(translate_first_word("i need water"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
hallo
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
guten
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
ich
